# Fine-tuning de un LLM básico: un ejemplo simple y de bajo costo

## Introducción

El fine-tuning adapta un modelo de lenguaje pre-entrenado a una tarea específica entrenándolo
adicionalmente en un conjunto de datos más pequeño y específico para la tarea. Este enfoque es
mucho más eficiente que entrenar un modelo de lenguaje grande (LLM) desde cero, requiriendo
mucho menos datos y computación.

En este tutorial:
- Usaremos **DistilGPT2** (82M parámetros) — una versión más pequeña y rápida de GPT-2.
- Haremos fine-tuning sobre **WikiText-2** — una colección de artículos de Wikipedia **en inglés**.
- Demostraremos **modelado de lenguaje causal** (predecir el siguiente token).
- Entrenaremos sobre un **subset de 2.000 ejemplos** para que la demo termine en minutos:
  **~3–6 min** en una GPU T4 de Google Colab y **~5–10 min** en Apple Silicon (MPS).

> Como DistilGPT2 y WikiText-2 están en inglés, el modelo se evalúa **en inglés**. Al final del
> notebook discutimos por qué: los datos definen la lengua y el dominio del modelo.

**Nota**: en Google Colab activa un runtime con GPU (Entorno de ejecución → Cambiar tipo de
entorno de ejecución → T4).


## 1. Configuración e instalación

En Colab, la siguiente celda instala las dependencias **fijadas** de la lección (el `torch`
preinstalado de Colab se reutiliza — ya viene compilado para su CUDA). En local no instala
nada: usa el entorno `uv` de la lección (ver README).


In [ ]:
# En Colab: instala las dependencias fijadas. Local: usa el entorno uv del README.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==5.13.1", "datasets==4.8.5", "accelerate==1.14.0",
    ])
    print("Dependencias instaladas (Colab)")
else:
    print("Entorno local: usando las dependencias del entorno uv (ver README)")


In [ ]:
import os

import torch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon (MPS) disponible")
else:
    device = "cpu"
    print("Sin GPU — el entrenamiento será lento (recomendado: Colab con T4)")

print(f"Dispositivo: {device}")


## 2. Cargar el dataset

Usaremos el dataset WikiText-2, que contiene artículos de Wikipedia y es perfecto para tareas
de modelado de lenguaje. Más adelante tomaremos un subset pequeño para que la demo sea rápida.


In [ ]:
from datasets import load_dataset

# WikiText-2 (versión de texto crudo), con su id canónico en el Hub
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(dataset)

# Inspeccionemos una muestra
print("\nTexto de muestra:")
print(dataset["train"][3]["text"][:200] + "...")

print("\nEstadísticas del dataset:")
print(f"Muestras de entrenamiento: {len(dataset['train'])}")
print(f"Muestras de validación: {len(dataset['validation'])}")
print(f"Muestras de prueba: {len(dataset['test'])}")


## 3. Inicializar el tokenizador y modelo pre-entrenado

Cargaremos DistilGPT2, una versión destilada de GPT-2 con aproximadamente 82 millones de
parámetros.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # GPT-2 destilado (pequeño, 82M parámetros)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Los modelos GPT-2 no tienen token de padding por defecto, así que usamos el EOS
tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo cargado: {model_name}")
print(f"Parámetros del modelo: {model.num_parameters():,}")
print(f"Tamaño del vocabulario: {len(tokenizer):,}")
print(f"Token de padding del tokenizador: {tokenizer.pad_token}")


## 4. Preprocesar los datos (tokenización)

### ¿Qué es un tokenizador?

Un **tokenizador** es un componente fundamental que convierte texto legible por humanos en una
secuencia de números (tokens) que el modelo puede procesar. Es como un traductor entre el
lenguaje humano y el lenguaje de la máquina.

#### Proceso de tokenización:
```
Texto original: "Hola mundo"
     ↓
Tokenización: ["Hola", " mundo"]
     ↓
IDs de tokens: [39, 23758]
     ↓
Tensores PyTorch: tensor([39, 23758])
```

#### Tipos de tokenización:
- **Nivel de palabra**: cada palabra es un token
- **Subpalabra (BPE/WordPiece)**: divide palabras en sub-unidades más pequeñas
- **Nivel de carácter**: cada carácter es un token

#### Componentes del tokenizador:
- **Vocabulario**: diccionario de tokens conocidos
- **Tokens especiales**: `<pad>`, `<unk>`, `<eos>`, etc.
- **Algoritmo de codificación**: cómo dividir el texto

### Ventajas de la tokenización por subpalabras:
- ✅ Maneja palabras fuera del vocabulario (OOV)
- ✅ Equilibrio entre flexibilidad y eficiencia
- ✅ Vocabulario más compacto que nivel de palabra
- ✅ Captura patrones morfológicos (prefijos, sufijos)

Necesitamos convertir las cadenas de texto en secuencias de IDs de tokens con las que el modelo
pueda entrenar. Antes, filtramos las entradas vacías y tomamos el **subset de la demo**.


In [ ]:
# Filtrar entradas vacías antes de tokenizar
def filter_empty_text(example):
    return len(example["text"].strip()) > 0


filtered_dataset = dataset.filter(filter_empty_text)
print("Dataset filtrado (sin entradas vacías):")
print(f"Train: {len(dataset['train'])} → {len(filtered_dataset['train'])}")
print(f"Validation: {len(dataset['validation'])} → {len(filtered_dataset['validation'])}")

# Subset para la demo: 2.000 ejemplos de train y 200 de validación.
# Con esto una época son ~250 pasos y la demo termina en minutos.
train_ds = filtered_dataset["train"].shuffle(seed=42).select(range(2000))
val_ds = filtered_dataset["validation"].shuffle(seed=42).select(range(200))
print(f"\nSubset de la demo — train: {len(train_ds)}, validation: {len(val_ds)}")


# Tokenizar: agrega 'input_ids' y 'attention_mask'
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)


tokenized_train = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])

print(f"\nPrimeros 10 IDs de tokens: {tokenized_train[0]['input_ids'][:10]}")
print(f"Muestra decodificada: {tokenizer.decode(tokenized_train[0]['input_ids'][:20])}")


## 5. Configurar el collator de datos

### ¿Qué es un data collator?

Un **data collator** es una función especial que toma un lote de ejemplos individuales y los
combina en un lote cohesivo para entrenar el modelo. Es como un organizador inteligente que:

#### Funciones principales:
1. **Padding dinámico**: rellena secuencias cortas para que todas tengan la misma longitud
2. **Preparación de etiquetas**: crea las etiquetas de entrenamiento automáticamente
3. **Conversión a tensores**: convierte listas a tensores PyTorch
4. **Manejo de lotes**: organiza múltiples ejemplos en un batch

#### Ejemplo visual del collator:
```
Entrada (3 secuencias de diferente longitud):
Ejemplo 1: [15, 23, 45]           (longitud: 3)
Ejemplo 2: [12, 67, 89, 34]       (longitud: 4)
Ejemplo 3: [88, 91]               (longitud: 2)

     ↓ COLLATOR APLICA PADDING ↓

Salida (batch con padding):
input_ids:     [[15, 23, 45, <pad>],
                [12, 67, 89, 34],
                [88, 91, <pad>, <pad>]]

attention_mask: [[1, 1, 1, 0],
                 [1, 1, 1, 1],
                 [1, 1, 0, 0]]

labels:        [[23, 45, <pad>, -100],
                [67, 89, 34, -100],
                [91, <pad>, -100, -100]]
```

#### Para modelado de lenguaje causal:
- **Input**: los primeros n-1 tokens
- **Labels**: los tokens desplazados una posición (siguientes tokens a predecir)
- **Padding token**: se ignora en la pérdida (label = -100)

### MLM vs. Causal LM:
- **MLM (BERT)**: enmascara tokens aleatoriamente, predice los enmascarados
- **Causal LM (GPT)**: predice el siguiente token secuencialmente


In [ ]:
from transformers import DataCollatorForLanguageModeling

# Collator de datos para modelado de lenguaje causal (mlm=False para modelos estilo GPT)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Collator de datos configurado para modelado de lenguaje causal")
print("MLM (modelado de lenguaje enmascarado): False")
print("Es decir, hacemos predicción del siguiente token (estilo GPT)")


## 6. Fine-tuning del modelo con Trainer

Usaremos la API `Trainer` de Hugging Face para agilizar el proceso de entrenamiento. El batch
por dispositivo se ajusta al hardware, pero con **acumulación de gradientes** el batch efectivo
es 8 en todos los casos — mismos resultados, distinta memoria.


In [ ]:
from transformers import TrainingArguments

# Batch por dispositivo + acumulación de gradientes: batch efectivo = 8 en todos los casos
if device == "cuda":
    per_device_batch_size, grad_accum_steps = 8, 1
elif device == "mps":
    per_device_batch_size, grad_accum_steps = 4, 2
else:
    per_device_batch_size, grad_accum_steps = 2, 4

training_args = TrainingArguments(
    output_dir="outputs/checkpoints",   # todo lo generado vive en outputs/ (git-ignorado)
    num_train_epochs=1,                 # 1 época para la demo (aumentar para mejores resultados)
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=per_device_batch_size,
    gradient_accumulation_steps=grad_accum_steps,
    eval_strategy="epoch",              # evaluar sobre validación al cierre de la época
    save_strategy="no",                 # sin checkpoints intermedios (ahorra tiempo y disco)
    logging_steps=25,
    report_to="none",
    warmup_steps=20,                    # calentamiento gradual de la tasa de aprendizaje
    learning_rate=5e-5,
)

print("Argumentos de entrenamiento configurados:")
print(f"- Épocas: {training_args.num_train_epochs}")
print(f"- Batch por dispositivo: {training_args.per_device_train_batch_size}")
print(f"- Acumulación de gradientes: {training_args.gradient_accumulation_steps}")
print(f"- Batch efectivo: {per_device_batch_size * grad_accum_steps}")
print(f"- Tasa de aprendizaje: {training_args.learning_rate}")


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,  # desde transformers v5, el tokenizador se pasa así (antes: tokenizer=)
)

print("¡Trainer inicializado!")
print(f"Ejemplos de entrenamiento: {len(tokenized_train)}")
print(f"Ejemplos de validación: {len(tokenized_val)}")


### Diagrama del bucle de fine-tuning

El fine-tuning sigue un proceso iterativo. Aquí está el flujo completo:

```
┌─────────────────────────────────────────────────────────────────┐
│                    BUCLE DE FINE-TUNING                        │
└─────────────────────────────────────────────────────────────────┘

    📚 DATASET                    🤖 MODELO PRE-ENTRENADO
    ─────────                     ──────────────────────
    WikiText-2                    DistilGPT2 (82M params)
         │                              │
         ▼                              ▼
    ┌─────────┐                   ┌──────────┐
    │Tokenizer│                   │  Model   │
    │  Input  │                   │ Weights  │
    └─────────┘                   └──────────┘
         │                              │
         ▼                              ▼
    ┌─────────────────────────────────────────┐
    │           DATA COLLATOR                 │
    │  • Agrupa ejemplos en batches          │
    │  • Aplica padding dinámico             │
    │  • Crea labels (siguiente token)       │
    └─────────────────────────────────────────┘
                     │
                     ▼
    ╔═════════════════════════════════════════╗
    ║         BUCLE DE ENTRENAMIENTO          ║
    ╚═════════════════════════════════════════╝
                     │
        ┌────────────┴────────────┐
        │                         │
        ▼                         ▼
    ┌─────────┐              ┌─────────┐
    │ FORWARD │              │BACKWARD │
    │  PASS   │              │  PASS   │
    └─────────┘              └─────────┘
        │                         │
        ▼                         ▼
    ┌─────────┐              ┌─────────┐
    │Predicción│              │Gradientes│
    │ Tokens  │              │Calculados│
    └─────────┘              └─────────┘
        │                         │
        ▼                         ▼
    ┌─────────┐              ┌─────────┐
    │ Cálculo │              │Optimizer│
    │  Loss   │──────────────▶│ Update  │
    └─────────┘              └─────────┘
                                  │
                                  ▼
                            ┌─────────┐
                            │ Nuevos  │
                            │ Pesos   │
                            └─────────┘
                                  │
                  ┌───────────────┴───────────────┐
                  │                              │
                  ▼                              ▼
              ¿Fin época?                   ¿Logging?
                  │                              │
                  ▼                              ▼
              ┌─────────┐                   ┌─────────┐
              │Evaluació│                   │ Print   │
              │n Val Set│                   │Progress │
              └─────────┘                   └─────────┘
```

### Detalles del proceso:

#### 1. **Forward pass** 🔄
```python
# El modelo recibe input_ids y attention_mask
logits = model(input_ids, attention_mask=attention_mask)
# Salida: probabilidades para cada token del vocabulario
```

#### 2. **Cálculo de loss** 📊
```python
# CrossEntropyLoss entre predicciones y tokens objetivo
loss = criterion(logits.view(-1, vocab_size), labels.view(-1))
```

#### 3. **Backward pass** 🔙
```python
# Calcula gradientes para cada parámetro
loss.backward()
```

#### 4. **Optimizer update** ⬆️
```python
# AdamW actualiza pesos según los gradientes
optimizer.step()
optimizer.zero_grad()
```

### Métricas clave:
- **Loss**: qué tan "equivocado" está el modelo
- **Perplexity**: exp(loss) — medida de "confusión" del modelo
- **Learning rate**: qué tan grandes son los pasos de actualización
- **Gradient norm**: magnitud de los gradientes (evita explosión)


In [ ]:
import gc
import time

print("Iniciando fine-tuning...")
print("~250 pasos: minutos en GPU/MPS, bastante más en CPU\n")

# Liberar memoria antes de entrenar (cortesía, útil en MPS)
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

inicio = time.time()
trainer.train()
minutos = (time.time() - inicio) / 60

print(f"\n¡Fine-tuning completado en {minutos:.1f} minutos!")


## 7. Probar el modelo fine-tuneado

Generemos texto a partir de varios prompts.

> **¿Por qué prompts en inglés?** DistilGPT2 fue pre-entrenado en inglés y lo afinamos con
> Wikipedia **en inglés**: 250 pasos sobre WikiText-2 no le enseñan español. Si lo evaluáramos
> con prompts en español obtendríamos texto incoherente — no porque el fine-tuning "haya
> fallado", sino porque **los datos definen la lengua y el dominio del modelo**. Para generar
> español haría falta partir de una base en español y un corpus en español (ver conclusiones).


In [ ]:
model.eval()

device_del_modelo = next(model.parameters()).device
print(f"Modelo en dispositivo: {device_del_modelo}")


def generate_text(prompt, max_new_tokens=60, temperature=0.8, top_k=50):
    """Generar la continuación de un prompt con el modelo fine-tuneado."""
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device_del_modelo) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


test_prompts = [
    "Once upon a time,",
    "The history of artificial intelligence",
    "In the field of computer science,",
    "Wikipedia is an encyclopedia that",
    "The development of technology has",
]

print("=== Generación de texto del modelo fine-tuneado ===\n")
for i, prompt in enumerate(test_prompts, 1):
    generated = generate_text(prompt)
    continuation = generated[len(prompt):].strip()
    print(f"Ejemplo {i}:")
    print(f'Prompt: "{prompt}"')
    print(f'Generado: "{continuation}"')
    print("-" * 60)


## 8. Comparar con el modelo original (opcional)

Carguemos el DistilGPT2 original y comparemos las salidas para ver el efecto del fine-tuning.
Con 1 época sobre un subset el cambio es sutil (más "tono Wikipedia"), pero visible.


In [ ]:
original_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
original_model.eval()
original_model = original_model.to(device_del_modelo)


def generate_with_original(prompt, max_new_tokens=60):
    """Generar texto con el modelo original (sin fine-tuning)."""
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device_del_modelo) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = original_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


comparison_prompt = "The history of artificial intelligence"

print("=== Comparación de modelos ===")
print(f'Prompt: "{comparison_prompt}"\n')

original_output = generate_with_original(comparison_prompt)
print("DistilGPT2 original:")
print(f'"{original_output[len(comparison_prompt):].strip()}"\n')

finetuned_output = generate_text(comparison_prompt)
print("Fine-tuneado en WikiText-2:")
print(f'"{finetuned_output[len(comparison_prompt):].strip()}"')


## 9. Guardar el modelo fine-tuneado (opcional)

Guardamos el modelo y el tokenizador en `outputs/` (git-ignorado). Desde transformers v5 el
formato de guardado es **safetensors**.


In [ ]:
ruta = "outputs/distilgpt2-finetuneado-demo"
model.save_pretrained(ruta)
tokenizer.save_pretrained(ruta)

print(f"Modelo fine-tuneado guardado en '{ruta}' (outputs/ está git-ignorado)")
print("Para cargarlo más tarde:")
print(f"model = AutoModelForCausalLM.from_pretrained('{ruta}')")
print(f"tokenizer = AutoTokenizer.from_pretrained('{ruta}')")


## 10. Conceptos clave y conclusiones

### Lo que logramos
1. **Cargamos un modelo pre-entrenado**: DistilGPT2 con 82M parámetros
2. **Preparamos datos de entrenamiento**: un subset de WikiText-2 (2.000 ejemplos)
3. **Hicimos fine-tuning del modelo**: con la API `Trainer` de Hugging Face
4. **Generamos texto**: probamos el modelo fine-tuneado y lo comparamos con el original

### Conceptos clave
- **Aprendizaje por transferencia**: partir de un modelo pre-entrenado y adaptarlo a nuevos datos
- **Modelado de lenguaje causal**: predecir el siguiente token de una secuencia
- **Tokenización**: convertir texto a tokens numéricos que el modelo puede procesar
- **Fine-tuning vs. entrenar desde cero**: mucho más eficiente y práctico

### Consideraciones de costo
- **Recursos gratuitos**: Google Colab (GPU T4) o tu propio Apple Silicon (MPS)
- **Modelos pequeños**: DistilGPT2 es manejable en hardware modesto
- **Entrenamiento limitado**: 1 época sobre un subset mantiene la demo en minutos

### Próximos pasos para mejorar resultados
1. **Más datos y más épocas**: usar el WikiText-2 completo (23.767 ejemplos) y 2–3 épocas
2. **Ajuste de hiperparámetros**: tasa de aprendizaje, batch, warmup, etc.
3. **Datos específicos del dominio**: usar un corpus relevante para tu caso de uso
4. **Modelos en español**: para generar español, partir de una base pre-entrenada en español
   y un corpus en español — la lengua del modelo la definen sus datos
5. **Técnicas avanzadas**: LoRA, QLoRA u otros métodos PEFT para afinar modelos más grandes
   con una fracción de la memoria

### Limitaciones de esta demo
- **Entrenamiento mínimo**: 1 época sobre 2.000 ejemplos (~250 pasos), por velocidad
- **Evaluación básica**: solo evaluación cualitativa (se podría agregar perplejidad, etc.)
- **Modelo y corpus en inglés**: el modelo resultante no genera español

Este tutorial proporciona una base para entender el fine-tuning de LLMs. ¡Con estos
fundamentos puedes explorar técnicas más avanzadas y modelos más grandes conforme crezcan tus
necesidades y recursos!
